In [2]:
# 1. Setting up and loading data
# Each row is one bar/restaurant open in April of a given year ("cohort").
# The target did_not_survive = 1 if, one year later, the venue had closed, become a
# different kind of business, or been replaced by a new business.
import json
import time
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import torch
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, confusion_matrix, log_loss, roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.neighbors import BallTree
from sklearn.preprocessing import OneHotEncoder, StandardScaler

SEED = 42                     # fixes all random choices, so every run gives the same result
np.random.seed(SEED)
torch.manual_seed(SEED)

DATA = Path("data/madrid_bars_restaurants.csv")
OUT = Path("artifacts")       # trained models and results are saved here for the app
OUT.mkdir(exist_ok=True)

df = pd.read_csv(DATA, dtype={"venue_id": str}, parse_dates=["terrace_permit_date"])
print(df.shape)
print(df.groupby("cohort").did_not_survive.agg(["count", "mean"]).round(3))

(33622, 17)
        count   mean
cohort              
2023    16762  0.065
2024    16860  0.096


In [3]:
# 2. Feature engineering
# Location: number of other venues within 250 m.
# Premises: shopping centre / market / street, not at street level, mixed use, venue type.
# Terrace: none / recent permit (<= 4 years) / established / unknown date.
# Bad coordinates (0 or far outside Madrid) are replaced by the barrio's average location.

MADRID_X, MADRID_Y = (420_000, 470_000), (4_450_000, 4_500_000)   # box around the city, in metres


def add_features(cohort_df):
    d = cohort_df.copy()
    snapshot = pd.Timestamp(f"{d.cohort.iloc[0]}-04-01")    # the day the features describe

    # fix broken coordinates
    ok = d.x.between(*MADRID_X) & d.y.between(*MADRID_Y)
    centre = d[ok].groupby("neighbourhood")[["x", "y"]].mean()
    for col in ["x", "y"]:
        d[col] = d[col].where(ok, d.neighbourhood.map(centre[col]))
        d[col] = d[col].fillna(d.loc[ok, col].median())

    # competitors: count neighbours within 250 m (minus 1 so a venue doesn't count itself)
    points = d[["x", "y"]].to_numpy()
    d["competitors_250m"] = BallTree(points).query_radius(points, r=250, count_only=True) - 1
    d["log_competitors_250m"] = np.log1p(d.competitors_250m)   # log: 0 vs 10 matters more than 100 vs 110

    # premises
    d["setting"] = d.location_type.where(d.location_type.isin(["street", "shopping centre", "market"]),
                                         "other")
    d["not_street_level"] = ((d.access == "inside a building") | (d.ground_floor == 0)).astype(int)
    d["mixed_use"] = (d.n_activities > d.n_food_activities).astype(int)

    # terrace: a recent permit suggests a younger business
    permit_age = (snapshot - d.terrace_permit_date).dt.days / 365.25
    d["terrace"] = np.select(
        [d.has_terrace == 0, permit_age.isna(), permit_age <= 4],
        ["none", "unknown date", "recent"], default="established")
    return d


# each year is processed on its own: competitors are counted among venues open that same year
df = pd.concat([add_features(g) for _, g in df.groupby("cohort")], ignore_index=True)

CATEGORICAL = ["district", "venue_type", "setting", "terrace"]
NUMERIC = ["log_competitors_250m", "not_street_level", "mixed_use"]
TARGET = "did_not_survive"
print(df[CATEGORICAL + NUMERIC].head())

  district      venue_type setting      terrace  log_competitors_250m  \
0   Centro  bar-restaurant  street  established              4.948760   
1   Centro       cafeteria  street       recent              4.276666   
2   Latina       cafeteria  street         none              1.609438   
3   Latina       cafeteria  street         none              3.465736   
4   Latina  bar-restaurant  street  established              2.397895   

   not_street_level  mixed_use  
0                 0          0  
1                 0          0  
2                 0          1  
3                 0          0  
4                 0          0  


In [ ]:
# 3. Split the data and prepare it for the model
# Train on the 2023 venues and test on the 2024 venues: the model is meant to score next
# year's venues, so we test it exactly that way. A random split would mix the two years.
train_full = df[df.cohort == 2023]
test = df[df.cohort == 2024].reset_index(drop=True)
train, val = train_test_split(train_full, test_size=0.2, stratify=train_full[TARGET],
                              random_state=SEED)
train, val = train.reset_index(drop=True), val.reset_index(drop=True)

preprocess = ColumnTransformer([
    ("cat", OneHotEncoder(handle_unknown="infrequent_if_exist", min_frequency=100,
                          sparse_output=False), CATEGORICAL),   # venue types with <100 rows are pooled
    ("num", StandardScaler(), NUMERIC),
])
X_train = preprocess.fit_transform(train).astype(np.float32)
X_val = preprocess.transform(val).astype(np.float32)
X_test = preprocess.transform(test).astype(np.float32)
y_train, y_val, y_test = (d[TARGET].to_numpy().astype(np.float32) for d in (train, val, test))
feature_names = [n.split("__", 1)[1] for n in preprocess.get_feature_names_out()]

print("train", X_train.shape, " validation", X_val.shape, " test", X_test.shape)
print(f"share that did not survive: train {y_train.mean():.3f}  val {y_val.mean():.3f}  "
      f"test {y_test.mean():.3f}")

train (13409, 42)  validation (3353, 42)  test (16860, 42)
share that did not survive: train 0.065  val 0.065  test 0.096


In [5]:
# 4. How we measure, and the naive baselines
# Only ~6.5% of venues fail, so accuracy is misleading ("everyone survives" is 93.5% accurate).
#   - ROC-AUC: does the model rank failing venues above surviving ones? 0.5 = random
#   - PR-AUC: the same, focused on the rare failing venues; compare it with the failure rate
#   - log loss: how good the probabilities are (lower is better)
# Baselines the model must beat:
#   - base rate: every venue gets the overall training failure rate
#   - district average: every venue gets its district's training failure rate
def evaluate(y, p):
    return {"roc_auc": roc_auc_score(y, p), "pr_auc": average_precision_score(y, p),
            "log_loss": log_loss(y, p)}


base_rate = y_train.mean()
district_rate = train.groupby("district")[TARGET].mean()


def baselines(d):
    return {"Baseline: base rate": np.full(len(d), base_rate),
            "Baseline: district average": d.district.map(district_rate).fillna(base_rate).to_numpy()}


for name, p in baselines(val).items():
    print(f"{name:28s}", {k: round(v, 4) for k, v in evaluate(y_val, p).items()})

Baseline: base rate          {'roc_auc': 0.5, 'pr_auc': 0.0653, 'log_loss': 0.2413}
Baseline: district average   {'roc_auc': 0.6053, 'pr_auc': 0.091, 'log_loss': 0.2375}


In [ ]:
# 5.logistic regression with L2 regularisation
# p = sigmoid(X @ w + b) gives each venue a probability of not surviving.
# Training minimises:  loss = binary cross-entropy + penalty * sum(w^2)
# The penalty keeps the weights small so the model doesn't overfit. Its strength comes from C
# (smaller C = stronger penalty). We write it as 1 / (2 * C * n) so that all three
# implementations solve exactly the same problem as scikit-learn.
def penalty_strength(C, n):
    return 1.0 / (2.0 * C * n)


def total_loss(X, y, w, b, C):
    # the loss above, computed in numpy: lets us check every version reached the same minimum
    z = X.astype(np.float64) @ w + b
    return float(np.mean(np.logaddexp(0, z) - y * z) + penalty_strength(C, len(y)) * w @ w)

In [7]:
# 6. Version 1: scikit-learn
# Try several values of C, keep the one with the lowest validation log loss, then refit.
C_VALUES = [0.003, 0.01, 0.03, 0.1, 0.3, 1.0, 3.0]
tuning = []
for C in C_VALUES:
    m = LogisticRegression(C=C, max_iter=5000, tol=1e-10).fit(X_train, y_train)
    tuning.append({"C": C, **evaluate(y_val, m.predict_proba(X_val)[:, 1])})
tuning = pd.DataFrame(tuning)
print(tuning.round(4).to_string(index=False))

C_BEST = float(tuning.loc[tuning.log_loss.idxmin(), "C"])
start = time.perf_counter()
sk_model = LogisticRegression(C=C_BEST, max_iter=5000, tol=1e-10).fit(X_train, y_train)
sk_seconds = time.perf_counter() - start
sk_w, sk_b = sk_model.coef_.ravel().astype(np.float64), float(sk_model.intercept_[0])
print(f"\nbest C = {C_BEST}   training loss = {total_loss(X_train, y_train, sk_w, sk_b, C_BEST):.6f}")

    C  roc_auc  pr_auc  log_loss
0.003   0.5590  0.0806    0.2405
0.010   0.5686  0.0866    0.2396
0.030   0.5795  0.0928    0.2386
0.100   0.5872  0.0948    0.2383
0.300   0.5893  0.0952    0.2385
1.000   0.5901  0.0958    0.2388
3.000   0.5902  0.0960    0.2389

best C = 0.1   training loss = 0.235386


In [8]:
# 7. Version 2: PyTorch, written by hand
# We create w and b ourselves and repeat four steps:
#   1. predict: p = sigmoid(X @ w + b)
#   2. loss: binary cross-entropy + penalty
#   3. loss.backward(): PyTorch computes the gradients (which way to move w and b)
#   4. update: w = w - learning_rate * gradient, inside torch.no_grad() so the update itself
#      isn't tracked, then reset the gradients to zero (PyTorch adds them up otherwise)
# Every step uses all training rows. We try a few learning rates and keep the one that
# reaches the lowest loss.
Xt, yt = torch.from_numpy(X_train), torch.from_numpy(y_train)
N_FEATURES = X_train.shape[1]
PENALTY = penalty_strength(C_BEST, len(y_train))
STEPS = 3000


def train_by_hand(lr):
    w = torch.zeros(N_FEATURES, requires_grad=True)
    b = torch.zeros(1, requires_grad=True)
    history = []
    for step in range(STEPS):
        p = torch.sigmoid(Xt @ w + b).clamp(1e-7, 1 - 1e-7)          # clamp: avoid log(0)
        bce = -(yt * torch.log(p) + (1 - yt) * torch.log(1 - p)).mean()
        loss = bce + PENALTY * (w @ w)
        loss.backward()
        with torch.no_grad():
            w -= lr * w.grad
            b -= lr * b.grad
        w.grad.zero_()
        b.grad.zero_()
        history.append(loss.item())
    return w.detach(), b.detach(), history


manual_tries = {}
for lr in [0.1, 0.5, 1.0, 2.0, 4.0]:
    manual_tries[lr] = train_by_hand(lr)[2][-1]
    print(f"learning rate {lr:<4} -> final loss {manual_tries[lr]:.6f}")
LR_MANUAL = min(manual_tries, key=lambda lr: manual_tries[lr] if np.isfinite(manual_tries[lr]) else np.inf)

start = time.perf_counter()
manual_w, manual_b, manual_history = train_by_hand(LR_MANUAL)
manual_seconds = time.perf_counter() - start
print(f"\nbest learning rate = {LR_MANUAL}   training loss = "
      f"{total_loss(X_train, y_train, manual_w.numpy().astype(np.float64), float(manual_b), C_BEST):.6f}")

learning rate 0.1  -> final loss 0.236789
learning rate 0.5  -> final loss 0.235610
learning rate 1.0  -> final loss 0.235449
learning rate 2.0  -> final loss 0.235391
learning rate 4.0  -> final loss 0.235386

best learning rate = 4.0   training loss = 0.235386


In [9]:
# 8. Version 3: PyTorch, the standard way
# Same model, but PyTorch's building blocks do the work:
#   - nn.Linear holds w and b, nn.Sigmoid turns the result into a probability
#   - nn.BCELoss is the binary cross-entropy
#   - the Adam optimiser does the update: zero_grad() and step() replace the manual steps
# The penalty is added by hand on w only, to match scikit-learn (which doesn't penalise b).
def build_model(n_features):
    return torch.nn.Sequential(torch.nn.Linear(n_features, 1), torch.nn.Sigmoid())


def train_standard(lr):
    model = build_model(N_FEATURES)
    torch.nn.init.zeros_(model[0].weight)            # start from zero, like the manual version
    torch.nn.init.zeros_(model[0].bias)
    loss_fn = torch.nn.BCELoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    history = []
    for step in range(STEPS):
        optimizer.zero_grad()
        p = model(Xt).squeeze(-1)
        loss = loss_fn(p, yt) + PENALTY * (model[0].weight ** 2).sum()
        loss.backward()
        optimizer.step()
        history.append(loss.item())
    return model, history


standard_tries = {}
for lr in [0.003, 0.01, 0.03, 0.1]:
    standard_tries[lr] = train_standard(lr)[1][-1]
    print(f"learning rate {lr:<6} -> final loss {standard_tries[lr]:.6f}")
LR_STANDARD = min(standard_tries, key=standard_tries.get)

start = time.perf_counter()
std_model, std_history = train_standard(LR_STANDARD)
std_seconds = time.perf_counter() - start
std_w = std_model[0].weight.detach().numpy().ravel().astype(np.float64)
std_b = float(std_model[0].bias.detach()[0])
print(f"\nbest learning rate = {LR_STANDARD}   training loss = "
      f"{total_loss(X_train, y_train, std_w, std_b, C_BEST):.6f}")

learning rate 0.003  -> final loss 0.236796
learning rate 0.01   -> final loss 0.235777
learning rate 0.03   -> final loss 0.235387
learning rate 0.1    -> final loss 0.235386

best learning rate = 0.1   training loss = 0.235386


In [10]:
# 9. Do the three versions agree?
# Same data and same loss, so they should end with the same weights and predictions.
# Tiny differences are normal: each optimiser stops at a slightly different point, and
# PyTorch uses less precise numbers (float32) than scikit-learn (float64).
def predict(w, b, X):
    return 1.0 / (1.0 + np.exp(-(X.astype(np.float64) @ w + b)))


models = {
    "scikit-learn": (sk_w, sk_b),
    "PyTorch by hand": (manual_w.numpy().astype(np.float64), float(manual_b)),
    "PyTorch standard": (std_w, std_b),
}
agreement = pd.DataFrame([{
    "model": name,
    "training_loss": total_loss(X_train, y_train, w, b, C_BEST),
    "max_weight_difference": np.max(np.abs(w - sk_w)),
    "max_prediction_difference": np.max(np.abs(predict(w, b, X_test) - predict(sk_w, sk_b, X_test))),
} for name, (w, b) in models.items()])
print(agreement.to_string(index=False))

weights = pd.DataFrame({name: w for name, (w, _) in models.items()}, index=feature_names)
print("\nlargest weights (positive = more likely to fail):")
print(weights.reindex(weights["scikit-learn"].abs().sort_values(ascending=False).index).head(10).round(3))

           model  training_loss  max_weight_difference  max_prediction_difference
    scikit-learn       0.235386               0.000000                   0.000000
 PyTorch by hand       0.235386               0.003763                   0.000299
PyTorch standard       0.235386               0.000735                   0.000148

largest weights (positive = more likely to fail):
                              scikit-learn  PyTorch by hand  PyTorch standard
district_Retiro                      0.732            0.732             0.732
district_Hortaleza                   0.593            0.593             0.593
district_Latina                     -0.592           -0.592            -0.592
district_Fuencarral-El Pardo         0.560            0.559             0.560
district_Carabanchel                -0.527           -0.527            -0.527
district_Chamartin                  -0.461           -0.462            -0.462
setting_other                       -0.406           -0.408            -0.4

In [11]:
# 10. Results: validation (2023) and test (2024), compared with the baselines
def all_predictions(d, X):
    preds = baselines(d)
    for name, (w, b) in models.items():
        preds[name] = predict(w, b, X)
    return preds


predictions = {"train": all_predictions(train, X_train), "validation": all_predictions(val, X_val),
               "test": all_predictions(test, X_test)}
results = pd.DataFrame([
    {"split": split, "model": name, **evaluate(y, p), "avg_predicted": p.mean(), "actual_rate": y.mean()}
    for split, y in [("validation", y_val), ("test", y_test)]
    for name, p in predictions[split].items()])
print(results.round(4).to_string(index=False))

     split                      model  roc_auc  pr_auc  log_loss  avg_predicted  actual_rate
validation        Baseline: base rate   0.5000  0.0653    0.2413         0.0653       0.0653
validation Baseline: district average   0.6053  0.0910    0.2375         0.0646       0.0653
validation               scikit-learn   0.5872  0.0948    0.2383         0.0647       0.0653
validation            PyTorch by hand   0.5872  0.0949    0.2383         0.0647       0.0653
validation           PyTorch standard   0.5872  0.0949    0.2383         0.0647       0.0653
      test        Baseline: base rate   0.5000  0.0962    0.3235         0.0653       0.0962
      test Baseline: district average   0.5475  0.1068    0.3268         0.0650       0.0962
      test               scikit-learn   0.5640  0.1104    0.3244         0.0643       0.0962
      test            PyTorch by hand   0.5640  0.1104    0.3244         0.0643       0.0962
      test           PyTorch standard   0.5640  0.1104    0.3244      

In [14]:
# 11. Decision threshold: the brewery asks for a deposit if risk >= threshold
COST_MISSED, COST_FALSE_ALARM = 1500.0, 150.0   # assumed: equipment lost vs. healthy venue lost to a rival


def cost(y, p, threshold):
    tn, fp, fn, tp = confusion_matrix(y, p >= threshold, labels=[0, 1]).ravel()
    return fn * COST_MISSED + fp * COST_FALSE_ALARM


thresholds = np.round(np.arange(0.01, 0.5, 0.005), 3)
val_costs = [cost(y_val, predictions["validation"]["scikit-learn"], t) for t in thresholds]
THRESHOLD = float(thresholds[int(np.argmin(val_costs))])
print(f"best threshold on validation: {THRESHOLD}")

decision = []
for split, y in [("validation", y_val), ("test", y_test)]:
    decision += [
        {"split": split, "policy": "deposit from nobody", "cost_eur": y.sum() * COST_MISSED},
        {"split": split, "policy": "deposit from everyone", "cost_eur": (1 - y).sum() * COST_FALSE_ALARM},
        {"split": split, "policy": f"model, threshold {THRESHOLD}",
         "cost_eur": cost(y, predictions[split]["scikit-learn"], THRESHOLD)},
    ]
decision = pd.DataFrame(decision)
print(decision.to_string(index=False))

best threshold on validation: 0.08
     split                policy  cost_eur
validation   deposit from nobody  328500.0
validation deposit from everyone  470100.0
validation model, threshold 0.08  310950.0
      test   deposit from nobody 2433000.0
      test deposit from everyone 2285700.0
      test model, threshold 0.08 2320650.0


In [15]:
# 12. Save everything the app needs (the app only loads these files, it never trains)
joblib.dump(sk_model, OUT / "model_sklearn.joblib")
joblib.dump(preprocess, OUT / "preprocess.joblib")
torch.save({"w": manual_w, "b": manual_b}, OUT / "model_pytorch_by_hand.pt")
torch.save(std_model.state_dict(), OUT / "model_pytorch_standard.pt")

columns = ["cohort", "venue_id", "name", "district", "neighbourhood", "x", "y", "outcome", TARGET,
           "competitors_250m"] + CATEGORICAL + NUMERIC
frames = []
for split, d in [("train", train), ("validation", val), ("test", test)]:
    frame = d[list(dict.fromkeys(columns))].copy()
    frame.insert(0, "split", split)
    for name, p in predictions[split].items():
        frame[f"p::{name}"] = p
    frames.append(frame)
pd.concat(frames, ignore_index=True).to_csv(OUT / "predictions.csv", index=False)

summary = {
    "features": {"categorical": CATEGORICAL, "numeric": NUMERIC, "encoded": feature_names},
    "rows": {"train": len(y_train), "validation": len(y_val), "test": len(y_test)},
    "C_tuning": tuning.to_dict("records"), "C_best": C_BEST,
    "learning_rate": {"by_hand": LR_MANUAL, "standard": LR_STANDARD, "steps": STEPS},
    "train_seconds": {"scikit-learn": sk_seconds, "PyTorch by hand": manual_seconds,
                      "PyTorch standard": std_seconds},
    "agreement": agreement.to_dict("records"),
    "weights": weights.reset_index(names="feature").to_dict("records"),
    "results": results.to_dict("records"),
    "costs": {"missed": COST_MISSED, "false_alarm": COST_FALSE_ALARM, "threshold": THRESHOLD,
              "decision": decision.to_dict("records")},
    "loss_history": {"PyTorch by hand": manual_history[::10], "PyTorch standard": std_history[::10]},
}
(OUT / "metrics.json").write_text(json.dumps(summary, indent=2, default=float))
print("saved:", sorted(f.name for f in OUT.iterdir()))

saved: ['metrics.json', 'model_pytorch_by_hand.pt', 'model_pytorch_standard.pt', 'model_sklearn.joblib', 'predictions.csv', 'preprocess.joblib']
